In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import copy
from torch import optim


from torch.utils.data import TensorDataset, DataLoader
import pickle
import numpy as np
import pandas as pd


class KANLayer(nn.Module):

  def __init__(self, in_features, out_features, num_basis=8):
    super().__init__()
    self.in_features = in_features
    self.out_features = out_features
    self.num_basis = num_basis

    centers = torch.linspace(-3.0, 3.0, num_basis).view(1, 1, num_basis)
    self.centers = nn.Parameter(centers.repeat(1, in_features, 1))

    self.log_widths = nn.Parameter(torch.zeros(1, in_features, num_basis))

    self.weight = nn.Parameter(torch.randn(in_features * num_basis, out_features) * 0.02)
    self.bias = nn.Parameter(torch.zeros(out_features))

  def forward(self, x):
    x_exp = x.unsqueeze(-1)

    widths = torch.exp(self.log_widths) + 1e-6
    basis = torch.exp(-((x_exp - self.centers) ** 2) / (2.0 * widths ** 2))

    phi = basis.reshape(x.size(0), self.in_features * self.num_basis)
    out = phi @ self.weight + self.bias
    return out

class ForwardKANNet(nn.Module):
  def __init__(self, in_dim=6, out_dim=3, num_basis=8):
    super().__init__()
    self.net = nn.Sequential(
        KANLayer(in_dim, 512, num_basis=num_basis),
        nn.SiLU(),
        nn.Dropout(0.2),
        KANLayer(512, 256, num_basis=num_basis),
        nn.SiLU(),
        nn.Dropout(0.2),
        KANLayer(256, 128, num_basis=num_basis),
        nn.SiLU(),
        nn.Dropout(0.2),
        KANLayer(128, out_dim, num_basis=num_basis)
    )

  def forward(self, x):
    return self.net(x)

def train_forward_net(forward_net, train_loader, val_loader, epochs = 100, device = 'cuda', lr = 1e-3):
  forward_net.to(device)
  opt = optim.AdamW(forward_net.parameters(), lr=lr)
  train_losses = []
  val_losses = []
  best_val_loss = float("inf")
  best_state = None

  for epoch in range(epochs):
    forward_net.train()
    total_train = 0.0
    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)
      opt.zero_grad()
      y_hat = forward_net(x)
      loss = F.mse_loss(y_hat, y)
      loss.backward()
      opt.step()
      total_train += loss.item()*x.size(0)
    train_epoch_loss = total_train / len(train_loader.dataset)
    train_losses.append(train_epoch_loss)
    forward_net.eval()
    total_val = 0.0
    for y, x in val_loader:
      y = y.to(device)
      x = x.to(device)

      y_hat = forward_net(x)
      val_loss = F.mse_loss(y_hat, y)
      total_val += val_loss.item() * x.size(0)

    val_epoch_loss = total_val / len(val_loader.dataset)
    val_losses.append(val_epoch_loss)

    if val_epoch_loss < best_val_loss:
      best_val_loss = val_epoch_loss
      best_state = copy.deepcopy(forward_net.state_dict())

    print(f"[Forward Net] Epoch: {epoch+1}, Train loss: {train_epoch_loss:.4f} Val loss: {val_epoch_loss:.4f}")
  if best_state is not None:
    forward_net.load_state_dict(best_state)
    print(f"Loaded best model with Val Loss: {best_val_loss:.4f}")
  return forward_net

class KAN_CVAE(nn.Module):
    def __init__(self, x_dim=6, cond_dim=3, latent_dim=30, num_basis=8, dropout=0.2):
      super().__init__()

      self.x_dim = x_dim
      self.cond_dim = cond_dim
      self.latent_dim = latent_dim

      self.enc1 = KANLayer(x_dim + cond_dim, 512, num_basis=num_basis)
      self.enc2 = KANLayer(512, 256, num_basis=num_basis)
      self.enc3 = KANLayer(256, 128, num_basis=num_basis)
      self.mu_head = KANLayer(128, latent_dim, num_basis=num_basis)
      self.logvar_head = KANLayer(128, latent_dim, num_basis=num_basis)
      self.dropout = nn.Dropout(dropout)

      self.dec1 = KANLayer(latent_dim + cond_dim, 128, num_basis=num_basis)
      self.dec2 = KANLayer(128, 256, num_basis=num_basis)
      self.dec3 = KANLayer(256, 512, num_basis=num_basis)
      self.out_head = KANLayer(512, x_dim, num_basis=num_basis)

    def encode(self, x, condition):
      h = torch.cat([x, condition], dim=1)
      h = self.dropout(F.silu(self.enc1(h)))
      h = self.dropout(F.silu(self.enc2(h)))
      h = self.dropout(F.silu(self.enc3(h)))
      mu = self.mu_head(h)
      logvar = self.logvar_head(h)
      return mu, logvar

    def reparameterize(self, mu, logvar):
      std = torch.exp(0.5 * logvar)
      eps = torch.randn_like(std)
      return mu + eps * std

    def decode(self, z, condition):
      h = torch.cat([z, condition], dim=1)
      h = self.dropout(F.silu(self.dec1(h)))
      h = self.dropout(F.silu(self.dec2(h)))
      h = self.dropout(F.silu(self.dec3(h)))
      return self.out_head(h)

    def forward(self, x, condition):
      mu, logvar = self.encode(x, condition)
      z = self.reparameterize(mu, logvar)
      recon = self.decode(z, condition)
      return recon, mu, logvar


def cvae_loss(recon_x, x, mu, logvar, forward_net, y, beta=0.001, lam_phys=5.0):
  MSE = F.mse_loss(recon_x, x, reduction='mean')
  KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
  base = MSE + beta*KLD

  y_hat = forward_net(recon_x)
  phys = F.mse_loss(y_hat, y, reduction='mean')

  return base  + lam_phys*phys, base.item(), phys.item()

def train_kan_cvae(cvae, forward_net, train_loader, val_loader, epochs=300, patience=10, beta=0.001, lam_phys=5.0, lr=1e-3, device="cuda"):

  cvae.to(device)
  forward_net.to(device)

  forward_net.eval()

  for p in forward_net.parameters():
    p.requires_grad = False

  optimizer = optim.AdamW(cvae.parameters(), lr=lr)
  scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.7)

  best_val_loss = float("inf")
  best_state = None
  no_improve_epochs = 0

  train_losses = []
  val_losses = []


  for epoch in range(epochs):
    cvae.train()
    train_total = 0.0
    train_base_total = 0.0
    train_phys_total = 0.0

    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)

      optimizer.zero_grad()

      recon_batch, mu, logvar = cvae(x, y)
      loss, base_loss, phys_loss = cvae_loss(
                recon_batch, x, mu, logvar,
                forward_net, y,
                beta=beta,
                lam_phys=lam_phys
            )

      loss.backward()
      optimizer.step()

      train_total += loss.item() * x.size(0)
      train_base_total += base_loss * x.size(0)
      train_phys_total += phys_loss *x.size(0)

    avg_train_loss = train_total / len(train_loader.dataset)
    avg_train_base = train_base_total / len(train_loader.dataset)
    avg_train_phys = train_phys_total / len(train_loader.dataset)
    train_losses.append(avg_train_loss)

    cvae.eval()
    val_total = 0.0
    val_base_total = 0.0
    val_phys_total = 0.0

    with torch.no_grad():
      for y, x in val_loader:
        y = y.to(device)
        x = x.to(device)

        recon_batch, mu, logvar = cvae(x, y)
        val_loss, val_base_loss, val_phys_loss = cvae_loss(
                        recon_batch, x, mu, logvar,
                        forward_net, y,
                        beta=beta,
                        lam_phys=lam_phys
                    )


        val_total += val_loss.item() * x.size(0)
        val_base_total += val_base_loss * x.size(0)
        val_phys_total += val_phys_loss * x.size(0)

    avg_val_loss = val_total / len(val_loader.dataset)
    avg_val_base = val_base_total / len(val_loader.dataset)
    avg_val_phys = val_phys_total / len(val_loader.dataset)
    val_losses.append(avg_val_loss)

    print(
                f"[PINN-KAN-CVAE] Epoch {epoch+1}, "
                f"Train Total: {avg_train_loss:.4f}, Train Base: {avg_train_base:.4f}, Train Phys: {avg_train_phys:.4f}, "
                f"Val Total: {avg_val_loss:.4f}, Val Base: {avg_val_base:.4f}, Val Phys: {avg_val_phys:.4f}"
            )

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      best_state = copy.deepcopy(cvae.state_dict())
      no_improve_epochs = 0
    else:
      no_improve_epochs += 1
      if no_improve_epochs >= patience:
        print(
                        f"Early stopping at epoch {epoch+1}. "
                        f"No improvement in validation loss for {patience} consecutive epochs."
                    )
        break
    scheduler.step()

  if best_state is not None:
    cvae.load_state_dict(best_state)
    print(f"Loaded best CVAE model with Val Loss: {best_val_loss:.4f}")
  return cvae

def generate_outputs(model, input_data, num_samples=10):
  model.eval()
  device = next(model.parameters()).device  # get model device

  with torch.no_grad():
    input_df = pd.DataFrame(
    input_data,columns=['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)'])
    input_data_scaled = Y_scaler.transform(input_df)

    conditions = torch.tensor(input_data_scaled, dtype=torch.float32, device=device)

    outputs = []
    for _ in range(num_samples):
      z = torch.randn(conditions.size(0), 30, device=device)
      output = model.decode(z, conditions)
      output = X_scaler.inverse_transform(output.cpu().numpy())
      outputs.append(output)

  return outputs

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_default_dtype(torch.float32)


device = 'cuda' if torch.cuda.is_available() else "cpu"

df_train = pd.read_csv('/content/train.csv')
df_val = pd.read_csv('/content/val.csv')


X_train, X_val = df_train.iloc[:, 0:6], df_val.iloc[:, 0:6]
y_train, y_val = df_train.iloc[:, 6:9], df_val.iloc[:, 6:9]

with open('/content/X_scaler_cvae.pkl', 'rb') as f:
  X_scaler = pickle.load(f)

with open('/content/Y_scaler_cvae.pkl', 'rb') as f:
  Y_scaler = pickle.load(f)


X_train_scaled = X_scaler.transform(X_train)
y_train_scaled = Y_scaler.transform(y_train)

X_val_scaled = X_scaler.transform(X_val)
y_val_scaled = Y_scaler.transform(y_val)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

train_data = TensorDataset(y_train_tensor, X_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

val_data = TensorDataset(y_val_tensor, X_val_tensor)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

forward_net = ForwardKANNet(in_dim=6, out_dim=3)
best_forward = train_forward_net(forward_net, train_loader, val_loader, epochs = 100, device = device)

cvae = KAN_CVAE(
    x_dim=6,
    cond_dim=3,
    latent_dim=30,
    num_basis=8,
    dropout=0.2
)

best_kan_cvae = train_kan_cvae(
    cvae,
    best_forward,
    train_loader,
    val_loader,
    epochs=300,
    patience=20,
    beta=0.001,
    lam_phys=0.5,
    lr=1e-3,
    device=device
)

df_test = pd.read_csv("/content/test.csv")

frequency, storage_modulus, loss_modulus = df_test['Frequency (Hz)'], df_test['Storage modulus (Pa)'], df_test['Loss modulus (Pa)']


# Initialize an empty DataFrame with specified columns
columns = ['Acrylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)',
          'Bottom Layer exposure time (s) ', 'Exposure time (s)', 'Frequency (Hz)',
          'Storage modulus (Pa)', 'Loss modulus (Pa)']

rows = []

for i, j, k in zip(frequency, storage_modulus, loss_modulus):
    input_data = [[i, j, k]]
    output_parameters  = generate_outputs(
    best_kan_cvae,
    input_data,
    num_samples=1,
)[0]
    combined_data = list(output_parameters[0]) + [i, j, k]
    rows.append(combined_data)

df = pd.DataFrame(rows, columns=columns)

df.to_csv('synth_kan_pinn_phys_2.0.csv', index=False)
print("\nSaved synthetic data!")

[Forward Net] Epoch: 1, Train loss: 1.0304 Val loss: 0.9919
[Forward Net] Epoch: 2, Train loss: 1.0218 Val loss: 0.9739
[Forward Net] Epoch: 3, Train loss: 1.0425 Val loss: 0.9753
[Forward Net] Epoch: 4, Train loss: 1.0296 Val loss: 0.9634
[Forward Net] Epoch: 5, Train loss: 1.0263 Val loss: 0.9770
[Forward Net] Epoch: 6, Train loss: 1.0367 Val loss: 0.9586
[Forward Net] Epoch: 7, Train loss: 1.0192 Val loss: 0.9625
[Forward Net] Epoch: 8, Train loss: 1.0200 Val loss: 0.9605
[Forward Net] Epoch: 9, Train loss: 1.0213 Val loss: 0.9675
[Forward Net] Epoch: 10, Train loss: 1.0182 Val loss: 1.0523
[Forward Net] Epoch: 11, Train loss: 1.0316 Val loss: 0.9964
[Forward Net] Epoch: 12, Train loss: 1.0169 Val loss: 0.9636
[Forward Net] Epoch: 13, Train loss: 1.0175 Val loss: 0.9852
[Forward Net] Epoch: 14, Train loss: 1.0428 Val loss: 0.9767
[Forward Net] Epoch: 15, Train loss: 1.0198 Val loss: 0.9656
[Forward Net] Epoch: 16, Train loss: 1.0213 Val loss: 0.9610
[Forward Net] Epoch: 17, Train lo